# Codebase Workflow Walkthrough

This notebook is a guided tour of the live `src/` code.

It is designed to answer three questions clearly:

1. What is the workflow of the code?
2. What does each layer do?
3. What is already implemented and working?

We start from the simplest direct interaction examples and move upward through the model layer, gauge metadata, gauge compilation, and the convention-fixed covariant and pure-gauge features.

It covers the main workflow of the codebase, but not every regression case that lives in `examples/examples.py`.

## Recommended way to read this notebook

Run the notebook from top to bottom.

The order is intentional:

- direct engine with scalars
- add derivatives
- add fermions and tensor structures
- mix sectors
- understand the metadata layer
- build a first `Model`
- compile gauge interactions from metadata
- compile covariant kinetic terms
- compile pure-gauge Yang-Mills structures

That mirrors the actual growth of the codebase.

Quick orientation while reading:

- when the notebook calls `vertex_factor(...)` or simplification helpers, we are using `src/symbolic/vertex_engine.py`
- when it creates `Field`, `InteractionTerm`, `Model`, or kinetic-term declarations, we are using the `src/model/` package
- when it calls `compile_minimal_gauge_interactions(...)`, we are using `src/compiler/gauge.py`
- when it calls `compile_covariant_terms(...)`, we are using `src/compiler/gauge.py`

In [33]:
import sys
from pathlib import Path
from fractions import Fraction
from pprint import pprint

root = Path.cwd()
src_path = root / "src"
if not src_path.exists():
    src_path = root.parent / "src"

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from symbolic.vertex_engine import (
    S,
    Expression,
    I,
    Delta,
    Dot,
    pcomp,
    vertex_factor,
    simplify_deltas,
    simplify_spinor_indices,
    simplify_vertex,
    infer_derivative_targets,
    compact_vertex_sum_form,
    compact_sum_notation,
)

from symbolic.spenso_structures import (
    SPINOR_KIND,
    LORENTZ_KIND,
    COLOR_FUND_KIND,
    COLOR_ADJ_KIND,
    gamma_matrix,
    gamma5_matrix,
    gauge_generator,
    structure_constant,
    lorentz_metric,
    simplify_gamma_chain,
    weak_gauge_generator, weak_structure_constant,
)

from lagrangian.operators import (
    psi_bar_gamma_psi,
    psi_bar_gamma5_psi,
    current_current,
    quark_gluon_current,
    scalar_gauge_contact,
)

from model import (
    Field,
    GaugeGroup,
    GaugeRepresentation,
    InteractionTerm,
    DerivativeAction,
    Model,
    Lagrangian,
    DeclaredLagrangian,
    CovD,
    PartialD,
    Gamma,
    FieldStrength,
    GaugeFixing,
    GhostLagrangian,
    SPINOR_INDEX,
    LORENTZ_INDEX,
    COLOR_FUND_INDEX,
    COLOR_ADJ_INDEX,
    WEAK_ADJ_INDEX,
    WEAK_FUND_INDEX,
)



from compiler.gauge import compile_minimal_gauge_interactions
from compiler.gauge import compile_covariant_terms

print("Python:", sys.executable)
print("src path:", src_path.resolve())

x = S("x")
d = S("d")


def direct_vertex(species_map=None, simplify_gamma=False, strip_externals=True, include_delta=False, **kwargs):
    expr = vertex_factor(
        x=x,
        d=d,
        strip_externals=strip_externals,
        include_delta=include_delta,
        **kwargs,
    )
    expr = simplify_deltas(expr, species_map=species_map)
    q_ = S("q_")
    x_ = S("x_")
    expr = expr.replace(Expression.EXP(-I * Dot(q_, x_)), 1)
    if simplify_gamma:
        expr = simplify_gamma_chain(expr)
    return expr


def model_vertex(interaction, external_legs, species_map=None, strip_externals=True, include_delta=False, simplify_gamma=False):
    expr = vertex_factor(
        interaction=interaction,
        external_legs=external_legs,
        x=x,
        d=d,
        strip_externals=strip_externals,
        include_delta=include_delta,
    )
    expr = simplify_deltas(expr, species_map=species_map)
    q_ = S("q_")
    x_ = S("x_")
    expr = expr.replace(Expression.EXP(-I * Dot(q_, x_)), 1)
    if simplify_gamma:
        expr = simplify_gamma_chain(expr)
    return expr


def show(title, expr):
    print()
    print(f"=== {title} ===")
    print(expr)


def labels(terms):
    return [term.label for term in terms]


Python: /Users/rems/Documents/MScThesis/.venv/bin/python
src path: /Users/rems/Documents/MScThesis/src


## 0. Workflow in one paragraph

The code has one central engine: `vertex_factor(...)` in `src/symbolic/vertex_engine.py`.

Around that engine, the current layers are:

- `src/symbolic/vertex_engine.py`: `vertex_factor(...)`, simplification helpers, derivative bookkeeping helpers
- `src/model/`: `Field`, `InteractionTerm`, `Model`, and the structured bridge into engine kwargs
- `src/compiler/gauge.py`: `compile_minimal_gauge_interactions(...)`
- `src/compiler/gauge.py`: `compile_covariant_terms(...)` and covariant-source expansion

So everything else either:

- prepares inputs for that engine, or
- builds symbolic tensor structures for the engine, or
- compiles higher-level physics declarations into ordinary `InteractionTerm` objects that the same engine can evaluate.

So the notebook always asks the same question: **what do we feed into `vertex_factor(...)`, and how do we build those inputs more systematically?**

In [34]:
p1, p2, p3, p4 = S("p1", "p2", "p3", "p4")
b1, b2, b3, b4 = S("b1", "b2", "b3", "b4")

phi0, chi0 = S("phi0", "chi0")
phiC0, phiCdag0 = S("phiC0", "phiCdag0")
phiQED0, phiQEDdag0 = S("phiQED0", "phiQEDdag0")
phiQCD0, phiQCDdag0 = S("phiQCD0", "phiQCDdag0")
psi0, psibar0 = S("psi0", "psibar0")
psiQED0, psibarQED0 = S("psiQED0", "psibarQED0")
A0, G0 = S("A0", "G0")

mu, nu, rho, sigma = S("mu", "nu", "rho", "sigma")
mu3, mu4 = S("mu3", "mu4")

lam4, g_sym, gD, yF, gV, eQED, qPsi, qPhi, gS = S(
    "lam4", "g", "gD", "yF", "gV", "eQED", "qPsi", "qPhi", "gS"
)

alpha_s = S("alpha_s")
a = S("a")
i_bar, i_psi = S("i_bar", "i_psi")
i1, i2, i3, i4 = S("i1", "i2", "i3", "i4")
s1, s2, s3, s4 = S("s1", "s2", "s3", "s4")
a3, a4, a5, a6 = S("a3", "a4", "a5", "a6")
c1, c2 = S("c1", "c2")

print("Common symbols initialised.")


Common symbols initialised.


## 1. Direct engine input with pure scalars

The most basic interface is the **direct API**.

We provide:

- `coupling`
- `alphas`: the field species appearing in the interaction monomial
- `betas`: the species attached to the external legs
- `ps`: the external momenta

For bosons this is already enough for the combinatorics.

In [35]:
L_phi4 = {
    "coupling": lam4,
    "alphas": [phi0, phi0, phi0, phi0],
    "betas": [b1, b2, b3, b4],
    "ps": [p1, p2, p3, p4],
}

L_phi2chi2 = {
    "coupling": g_sym,
    "alphas": [phi0, phi0, chi0, chi0],
    "betas": [b1, b2, b3, b4],
    "ps": [p1, p2, p3, p4],
}

print("Direct API dictionary for phi^4:")
pprint(L_phi4)

V_phi4_full = vertex_factor(x=x, d=d, **L_phi4)
V_phi4_reduced = direct_vertex(
    **L_phi4,
    species_map={b1: phi0, b2: phi0, b3: phi0, b4: phi0},
)

V_phi2chi2 = direct_vertex(
    **L_phi2chi2,
    species_map={b1: phi0, b2: phi0, b3: chi0, b4: chi0},
)

show("phi^4 with overall momentum delta", V_phi4_full)
show("phi^4 reduced vertex", V_phi4_reduced)
show("phi^2 chi^2 reduced vertex", V_phi2chi2)


Direct API dictionary for phi^4:
{'alphas': [phi0, phi0, phi0, phi0],
 'betas': [b1, b2, b3, b4],
 'coupling': lam4,
 'ps': [p1, p2, p3, p4]}

=== phi^4 with overall momentum delta ===
24𝑖*lam4*(2*𝜋)^d*delta(b1,phi0)*delta(b2,phi0)*delta(b3,phi0)*delta(b4,phi0)*Delta(p1+p2+p3+p4)

=== phi^4 reduced vertex ===
24𝑖*lam4

=== phi^2 chi^2 reduced vertex ===
4𝑖*g


## 2. Add derivatives

Derivative interactions are encoded by:

- `derivative_indices`: which Lorentz indices appear
- `derivative_targets`: which field slots those derivatives act on

The helper `infer_derivative_targets(...)` builds these lists from a more readable slot-based description.

In [36]:
deriv_indices, deriv_targets = infer_derivative_targets([(0, [mu]), (1, [nu])])

L_deriv = {
    "coupling": gD,
    "alphas": [phi0, phi0, phi0, phi0],
    "betas": [b1, b2, b3, b4],
    "ps": [p1, p2, p3, p4],
    "derivative_indices": deriv_indices,
    "derivative_targets": deriv_targets,
}

V_deriv = direct_vertex(
    **L_deriv,
    species_map={b1: phi0, b2: phi0, b3: phi0, b4: phi0},
)

V_deriv_compact = compact_vertex_sum_form(
    coupling=gD,
    ps=[p1, p2, p3, p4],
    derivative_indices=deriv_indices,
    derivative_targets=deriv_targets,
    d=d,
    field_species=[phi0, phi0, phi0, phi0],
    leg_species=[phi0, phi0, phi0, phi0],
)

print("Derivative indices:", deriv_indices)
print("Derivative targets:", deriv_targets)
print("Compact sigma notation:", compact_sum_notation(derivative_indices=deriv_indices, derivative_targets=deriv_targets, n_legs=4))

show("Expanded derivative vertex", V_deriv)
show("Compact derivative sum form", V_deriv_compact)


Derivative indices: [mu, nu]
Derivative targets: [0, 1]
Compact sigma notation: (2)! * Σ_{a, b distinct} p_{a,mu} p_{b,nu}

=== Expanded derivative vertex ===
1𝑖*(-2*gD*pcomp(p1,mu)*pcomp(p2,nu)-2*gD*pcomp(p1,mu)*pcomp(p3,nu)-2*gD*pcomp(p1,mu)*pcomp(p4,nu)-2*gD*pcomp(p1,nu)*pcomp(p2,mu)-2*gD*pcomp(p1,nu)*pcomp(p3,mu)-2*gD*pcomp(p1,nu)*pcomp(p4,mu)-2*gD*pcomp(p2,mu)*pcomp(p3,nu)-2*gD*pcomp(p2,mu)*pcomp(p4,nu)-2*gD*pcomp(p2,nu)*pcomp(p3,mu)-2*gD*pcomp(p2,nu)*pcomp(p4,mu)-2*gD*pcomp(p3,mu)*pcomp(p4,nu)-2*gD*pcomp(p3,nu)*pcomp(p4,mu))

=== Compact derivative sum form ===
-1𝑖*gD*(2*𝜋)^d*(2*pcomp(p1,mu)*pcomp(p2,nu)+2*pcomp(p1,mu)*pcomp(p3,nu)+2*pcomp(p1,mu)*pcomp(p4,nu)+2*pcomp(p1,nu)*pcomp(p2,mu)+2*pcomp(p1,nu)*pcomp(p3,mu)+2*pcomp(p1,nu)*pcomp(p4,mu)+2*pcomp(p2,mu)*pcomp(p3,nu)+2*pcomp(p2,mu)*pcomp(p4,nu)+2*pcomp(p2,nu)*pcomp(p3,mu)+2*pcomp(p2,nu)*pcomp(p4,mu)+2*pcomp(p3,mu)*pcomp(p4,nu)+2*pcomp(p3,nu)*pcomp(p4,mu))*Delta(p1+p2+p3+p4)


## 3. Add fermions and tensor structures

For fermions we must tell the engine more:

- the statistics are fermionic
- the field roles are `psibar`, `psi`, `scalar`, `vector`, ...
- the spinor labels must be explicit if we want open spinor-index output

This is where the direct engine already starts to look like a real Feynman-rule generator.

In [37]:
L_yukawa = {
    "coupling": yF,
    "alphas": [psibar0, psi0, phi0],
    "betas": [b1, b2, b3],
    "ps": [p1, p2, p3],
    "statistics": "fermion",
    "field_roles": ["psibar", "psi", "scalar"],
    "leg_roles": ["psibar", "psi", "scalar"],
    "field_spinor_indices": [a,a, None],
    "leg_spins": [s1, s2, s3],
}

V_yukawa = direct_vertex(
    **L_yukawa,
    species_map={b1: psibar0, b2: psi0, b3: phi0},
)

L_vec_current = {
    "coupling": gV * gamma_matrix(i_bar, i_psi, mu),
    "alphas": [psibar0, psi0, A0],
    "betas": [b1, b2, b3],
    "ps": [p1, p2, p3],
    "statistics": "fermion",
    "field_roles": ["psibar", "psi", "vector"],
    "leg_roles": ["psibar", "psi", "vector"],
    "field_index_labels": [
        {SPINOR_KIND: i_bar},
        {SPINOR_KIND: i_psi},
        {LORENTZ_KIND: mu},
    ],
    "leg_index_labels": [
        {SPINOR_KIND: i1},
        {SPINOR_KIND: i2},
        {LORENTZ_KIND: mu3},
    ],
    "leg_spins": [s1, s2, s3],
}

V_vec_current = direct_vertex(
    **L_vec_current,
    species_map={b1: psibar0, b2: psi0, b3: A0},
    simplify_gamma=True,
)

show("Yukawa vertex from direct API", V_yukawa)
show("Vector current with explicit open indices", V_vec_current)



=== Yukawa vertex from direct API ===
1𝑖*yF*g(bis(4, i1),bis(4, i2))

=== Vector current with explicit open indices ===
1𝑖*gV*gamma(bis(4, i1),bis(4, i2),mink(4, mu3))


## 4. Mix fermions and derivatives

The engine is not limited to pure scalar derivatives or pure fermion bilinears.

It can also handle mixed operators such as

$$ y_F \, \bar\psi\psi\,(\partial_\mu \phi)(\partial_\nu \chi). $$

In [38]:
L_mix = {
    "coupling": yF,
    "alphas": [psibar0, psi0, phi0, chi0],
    "betas": [b1, b2, b3, b4],
    "ps": [p1, p2, p3, p4],
    "derivative_indices": [mu, nu],
    "derivative_targets": [2, 3],
    "statistics": "fermion",
    "field_roles": ["psibar", "psi", "scalar", "scalar"],
    "leg_roles": ["psibar", "psi", "scalar", "scalar"],
    "field_spinor_indices": [alpha_s, alpha_s, None, None],
    "leg_spins": [s1, s2, s3, s4],
}

V_mix = direct_vertex(
    **L_mix,
    species_map={b1: psibar0, b2: psi0, b3: phi0, b4: chi0},
)

show("Mixed fermion-scalar derivative operator", V_mix)



=== Mixed fermion-scalar derivative operator ===
-1𝑖*yF*g(bis(4, i1),bis(4, i2))*pcomp(p3,mu)*pcomp(p4,nu)


## 5. First metadata objects: `Field`, `FieldOccurrence`, `ExternalLeg`, `InteractionTerm`

The direct API is useful, but the codebase really wants to move toward structured model input.

The core metadata objects are:

- `Field`: one declared particle field
- `FieldOccurrence`: one appearance of that field inside an interaction term
- `ExternalLeg`: one external state used when evaluating a vertex
- `InteractionTerm`: one monomial in the Lagrangian

The key bridge method is `InteractionTerm.to_vertex_kwargs(...)`: it translates the structured model layer into the parallel-list format expected by the engine.

In [39]:
PhiField = Field("Phi", spin=0, self_conjugate=True, symbol=phi0)
ChiField = Field("Chi", spin=0, self_conjugate=True, symbol=chi0)
PsiField = Field(
    "Psi",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=psi0,
    conjugate_symbol=psibar0,
    indices=(SPINOR_INDEX,),
)
GaugeField = Field("A", spin=1, self_conjugate=True, symbol=A0, indices=(LORENTZ_INDEX,))

phi_occ = PhiField.occurrence()
psibar_occ = PsiField.occurrence(conjugated=True, labels={SPINOR_KIND: alpha_s})
psi_leg = PsiField.leg(p2, spin=s2, labels={SPINOR_KIND: i2})

print("Example FieldOccurrence:", psibar_occ)
print("  role   =", psibar_occ.role)
print("  species=", psibar_occ.species)
print()
print("Example ExternalLeg:", psi_leg)
print("  role            =", psi_leg.role)
print("  effective species=", psi_leg.effective_species)

TERM_phi4 = InteractionTerm(
    coupling=lam4,
    fields=tuple(PhiField.occurrence() for _ in range(4)),
    label="lam4 * phi^4",
)
LEGS_phi4 = tuple(PhiField.leg(p, species=b) for p, b in [(p1, b1), (p2, b2), (p3, b3), (p4, b4)])

TERM_yukawa = InteractionTerm(
    coupling=yF,
    fields=(
        PsiField.occurrence(conjugated=True, labels={SPINOR_KIND: alpha_s}),
        PsiField.occurrence(labels={SPINOR_KIND: alpha_s}),
        PhiField.occurrence(),
    ),
    label="yF * psibar psi phi",
)

LEGS_yukawa = (
    PsiField.leg(p1, conjugated=True, spin=s1, labels={SPINOR_KIND: i1}),
    PsiField.leg(p2, spin=s2, labels={SPINOR_KIND: i2}),
    PhiField.leg(p3, species=b3),
)

print("Keys produced by InteractionTerm.to_vertex_kwargs(...):")
pprint(sorted(TERM_yukawa.to_vertex_kwargs(LEGS_yukawa).keys()))

V_phi4_model = model_vertex(
    TERM_phi4,
    LEGS_phi4,
    species_map={b1: phi0, b2: phi0, b3: phi0, b4: phi0},
)

V_yukawa_model = model_vertex(
    TERM_yukawa,
    LEGS_yukawa,
    species_map={b3: phi0},
)

show("phi^4 through the model layer", V_phi4_model)
show("Yukawa through the model layer", V_yukawa_model)

assert V_yukawa_model.expand().to_canonical_string() == V_yukawa.expand().to_canonical_string()
print("\
Direct Yukawa and model-layer Yukawa agree.")


Example FieldOccurrence: FieldOccurrence(field=Field(name='Psi', spin=Fraction(1, 2), self_conjugate=False, indices=(IndexType(name='Spinor', representation=Representation { rep: SelfDual(1), dim: Concrete(4) }, kind='spinor', prefix='i'),), kind='fermion', statistics='fermion', symbol=psi0, conjugate_symbol=psibar0, mass=None, quantum_numbers={}), conjugated=True, labels={'spinor': alpha_s})
  role   = FieldRole('psibar')
  species= psibar0

Example ExternalLeg: ExternalLeg(field=Field(name='Psi', spin=Fraction(1, 2), self_conjugate=False, indices=(IndexType(name='Spinor', representation=Representation { rep: SelfDual(1), dim: Concrete(4) }, kind='spinor', prefix='i'),), kind='fermion', statistics='fermion', symbol=psi0, conjugate_symbol=psibar0, mass=None, quantum_numbers={}), momentum=p2, conjugated=False, species=None, spin=s2, labels={'spinor': i2})
  role            = FieldRole('psi')
  effective species= psi0
Keys produced by InteractionTerm.to_vertex_kwargs(...):
['alphas',
 'b

## 6. First `Model` object

A `Model` is the top-level container for a theory.

At minimum it can already store:

- fields
- explicit interactions
- gauge groups
- later: covariant terms and gauge kinetic terms

In [40]:
ToyScalarModel = Model(
    name="Toy scalar model",
    fields=(PhiField, ChiField),
    interactions=(TERM_phi4,),
)

print(ToyScalarModel)
print()
print("find_field('Phi') ->", ToyScalarModel.find_field("Phi"))
print("first stored interaction label ->", ToyScalarModel.interactions[0].label)


Model(name='Toy scalar model', gauge_groups=(), fields=(Field(name='Phi', spin=0, self_conjugate=True, indices=(), kind='scalar', statistics='boson', symbol=phi0, conjugate_symbol=None, mass=None, quantum_numbers={}), Field(name='Chi', spin=0, self_conjugate=True, indices=(), kind='scalar', statistics='boson', symbol=chi0, conjugate_symbol=None, mass=None, quantum_numbers={})), parameters=(), interactions=(InteractionTerm(coupling=lam4, fields=(FieldOccurrence(field=Field(name='Phi', spin=0, self_conjugate=True, indices=(), kind='scalar', statistics='boson', symbol=phi0, conjugate_symbol=None, mass=None, quantum_numbers={}), conjugated=False, labels={}), FieldOccurrence(field=Field(name='Phi', spin=0, self_conjugate=True, indices=(), kind='scalar', statistics='boson', symbol=phi0, conjugate_symbol=None, mass=None, quantum_numbers={}), conjugated=False, labels={}), FieldOccurrence(field=Field(name='Phi', spin=0, self_conjugate=True, indices=(), kind='scalar', statistics='boson', symbol=

## 7. Gauge metadata: `GaugeRepresentation` and `GaugeGroup`

To move beyond handwritten interaction terms, the code introduces gauge metadata.

- `GaugeRepresentation` tells the compiler how to build the generator tensor for one matter representation
- `GaugeGroup` defines the symmetry itself: coupling, gauge boson, structure constants, charge label, and supported matter representations

This is the basis for the gauge compiler.

In [41]:
PhiQEDField = Field(
    "PhiQED",
    spin=0,
    self_conjugate=False,
    symbol=phiQED0,
    conjugate_symbol=phiQEDdag0,
    quantum_numbers={"Q": qPhi},
)

PsiQEDField = Field(
    "PsiQED",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=psiQED0,
    conjugate_symbol=psibarQED0,
    indices=(SPINOR_INDEX,),
    quantum_numbers={"Q": qPsi},
)

PhiQCDField = Field(
    "PhiQCD",
    spin=0,
    self_conjugate=False,
    symbol=phiQCD0,
    conjugate_symbol=phiQCDdag0,
    indices=(COLOR_FUND_INDEX,),
)

QuarkField = Field(
    "q",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=psi0,
    conjugate_symbol=psibar0,
    indices=(SPINOR_INDEX, COLOR_FUND_INDEX),
)

GluonField = Field(
    "G",
    spin=1,
    self_conjugate=True,
    symbol=G0,
    indices=(LORENTZ_INDEX, COLOR_ADJ_INDEX),
)

COLOR_FUND_REP = GaugeRepresentation(
    index=COLOR_FUND_INDEX,
    generator_builder=gauge_generator,
    name="fundamental",
)

QED_GROUP = GaugeGroup(
    name="U1QED",
    abelian=True,
    coupling=eQED,
    gauge_boson="A",
    charge="Q",
)

QCD_GROUP = GaugeGroup(
    name="SU3C",
    abelian=False,
    coupling=gS,
    gauge_boson=G0,
    structure_constant=structure_constant,
    representations=(COLOR_FUND_REP,),
)

MODEL_QED_BASE = Model(
    name="FermionQED-minimal",
    gauge_groups=(QED_GROUP,),
    fields=(PsiQEDField, GaugeField),
)

MODEL_QCD_BASE = Model(
    name="QCD-minimal",
    gauge_groups=(QCD_GROUP,),
    fields=(QuarkField, GluonField),
)

print("QCD representation seen by the quark ->", QCD_GROUP.matter_representation(QuarkField))
print("Gauge boson resolved for QED ->", MODEL_QED_BASE.gauge_boson_field(QED_GROUP))


QCD representation seen by the quark -> GaugeRepresentation(index=IndexType(name='ColorFund', representation=Representation { rep: Dualizable(3), dim: Concrete(3) }, kind='color_fund', prefix='c'), generator_builder=<function gauge_generator at 0x108db4930>, name='fundamental', slot=None, slot_policy='unique')
Gauge boson resolved for QED -> Field(name='A', spin=1, self_conjugate=True, indices=(IndexType(name='Lorentz', representation=Representation { rep: InlineMetric(0), dim: Concrete(4) }, kind='lorentz', prefix='mu'),), kind='vector', statistics='boson', symbol=A0, conjugate_symbol=None, mass=None, quantum_numbers={})


## 8. Minimal gauge compiler

The **minimal gauge compiler** turns gauge metadata into ordinary `InteractionTerm` objects.

It is a structural compiler: it inserts the right charges, generators, and spectator identities, but it is not yet the convention-fixed covariant expansion of full kinetic terms.

This section uses:

- `GaugeRepresentation`, `GaugeGroup`, `Model`, and `InteractionTerm` from the `src/model/` package
- `compile_minimal_gauge_interactions(...)` from `src/compiler/gauge.py`

In [42]:
compiled_qed = compile_minimal_gauge_interactions(MODEL_QED_BASE)
compiled_qcd = compile_minimal_gauge_interactions(MODEL_QCD_BASE)

print("Minimal QED terms:", labels(compiled_qed))
print("Minimal QCD terms:", labels(compiled_qcd))

LEGS_qed = (
    PsiQEDField.leg(p1, conjugated=True, spin=s1, labels={SPINOR_KIND: i1}),
    PsiQEDField.leg(p2, spin=s2, labels={SPINOR_KIND: i2}),
    GaugeField.leg(p3, labels={LORENTZ_KIND: mu3}),
)

LEGS_quark_gluon = (
    QuarkField.leg(p1, conjugated=True, spin=s1, labels={SPINOR_KIND: i1, COLOR_FUND_KIND: c1}),
    QuarkField.leg(p2, spin=s2, labels={SPINOR_KIND: i2, COLOR_FUND_KIND: c2}),
    GluonField.leg(p3, labels={LORENTZ_KIND: mu3, COLOR_ADJ_KIND: a3}),
)

V_qed_min = model_vertex(compiled_qed[0], LEGS_qed, simplify_gamma=True)
V_qcd_min = model_vertex(compiled_qcd[0], LEGS_quark_gluon, simplify_gamma=True)

show("Minimal gauge compiler: fermion QED current", V_qed_min)
show("Minimal gauge compiler: quark-gluon current", V_qcd_min)


Minimal QED terms: ['U1QED: PsiQED gauge current']
Minimal QCD terms: ['SU3C: q gauge current']

=== Minimal gauge compiler: fermion QED current ===
-1𝑖*eQED*qPsi*gamma(bis(4, i1),bis(4, i2),mink(4, mu3))

=== Minimal gauge compiler: quark-gluon current ===
-1𝑖*gS*gamma(bis(4, i1),bis(4, i2),mink(4, mu3))*t(coad(8, a3),cof(3, c1),cof(3, c2))


## 8.5 Repeated same-kind index slots

The current code can now handle fields that carry the same index kind more than once, as long as the active gauge slot is declared explicitly.

This uses:

- `Field(...)` and `GaugeRepresentation(slot=...)` from the `src/model/` package
- `compile_minimal_gauge_interactions(...)` from `src/compiler/gauge.py`

If the representation is ambiguous, the compiler now rejects the model instead of silently collapsing repeated slots.

In [43]:
phiBi0, phiBidag0 = S("phiBi0", "phiBidag0")
c3, c4 = S("c3", "c4")

PhiBiField = Field(
    "PhiBi",
    spin=0,
    self_conjugate=False,
    symbol=phiBi0,
    conjugate_symbol=phiBidag0,
    indices=(COLOR_FUND_INDEX, COLOR_FUND_INDEX),
)

COLOR_FUND_REP_SLOT0 = GaugeRepresentation(
    index=COLOR_FUND_INDEX,
    generator_builder=gauge_generator,
    name="fundamental_slot0",
    slot=0,
)

QCD_GROUP_BISLOT = GaugeGroup(
    name="SU3CBi",
    abelian=False,
    coupling=gS,
    gauge_boson=G0,
    structure_constant=structure_constant,
    representations=(COLOR_FUND_REP_SLOT0,),
)

QCD_GROUP_AMBIGUOUS = GaugeGroup(
    name="SU3CAmbiguous",
    abelian=False,
    coupling=gS,
    gauge_boson=G0,
    structure_constant=structure_constant,
    representations=(COLOR_FUND_REP,),
)

MODEL_BISLOT_OK = Model(
    name="ScalarQCD-bislot-minimal",
    gauge_groups=(QCD_GROUP_BISLOT,),
    fields=(PhiBiField, GluonField),
)

MODEL_BISLOT_BAD = Model(
    name="ScalarQCD-bislot-ambiguous",
    gauge_groups=(QCD_GROUP_AMBIGUOUS,),
    fields=(PhiBiField, GluonField),
)

try:
    compile_minimal_gauge_interactions(MODEL_BISLOT_BAD)
except ValueError as exc:
    print("Ambiguous repeated-slot model rejected:")
    print(" ", exc)

compiled_bislot = compile_minimal_gauge_interactions(MODEL_BISLOT_OK)
print("Repeated-slot compiled labels:", labels(compiled_bislot))

LEGS_bislot = (
    PhiBiField.leg(p1, conjugated=True, species=b1, labels={COLOR_FUND_KIND: (c1, c3)}),
    PhiBiField.leg(p2, species=b2, labels={COLOR_FUND_KIND: (c2, c4)}),
    GluonField.leg(p3, labels={LORENTZ_KIND: mu3, COLOR_ADJ_KIND: a3}, species=b3),
)

V_bislot = (
    model_vertex(compiled_bislot[0], LEGS_bislot, species_map={b1: phiBidag0, b2: phiBi0, b3: G0})
    + model_vertex(compiled_bislot[1], LEGS_bislot, species_map={b1: phiBidag0, b2: phiBi0, b3: G0})
)

show("Repeated-slot scalar gauge current (slot 1 active, slot 2 spectator)", V_bislot)


Ambiguous repeated-slot model rejected:
  Field 'PhiBi' carries repeated index type 'ColorFund'; declare GaugeRepresentation(slot=...) for strict selection, or set GaugeRepresentation(slot_policy='sum') to sum over all matching slots.
Repeated-slot compiled labels: ['SU3CBi: scalar current (+)_slot1', 'SU3CBi: scalar current (-)_slot1', 'SU3CBi: scalar contact [slots 1,1]']

=== Repeated-slot scalar gauge current (slot 1 active, slot 2 spectator) ===
-1𝑖*gS*g(cof(3, c3),cof(3, c4))*t(coad(8, a3),cof(3, c1),cof(3, c2))*pcomp(p1,mu)+1𝑖*gS*g(cof(3, c3),cof(3, c4))*t(coad(8, a3),cof(3, c1),cof(3, c2))*pcomp(p2,mu)


## 9. Declarative kinetic sectors via `lagrangian_decl`

The recommended front-end is now one unified declaration entry point:

- `Model(..., lagrangian_decl=...)`

For the standard gauge-generated pieces, use canonical declarative objects:

- `I * Psi.bar * Gamma(mu) * CovD(Psi, mu)`
- `CovD(Phi.bar, mu) * CovD(Phi, mu)`
- `-(1/4) * FieldStrength(G, mu, nu) * FieldStrength(G, mu, nu)`

For ordinary local derivative operators, use `PartialD(...)` inside the same
source monomial language.

The important implementation point is now simple: `lagrangian_decl` stores source
terms, and `Model.lagrangian()` inspects them one by one. Pure local monomials are
turned directly into `InteractionTerm`s, while terms containing canonical `CovD`,
`FieldStrength`, gauge-fixing, or ghost structure are expanded through the compiler.


In [44]:
mu_decl = S("mu_decl")

MODEL_QED_FERMION_COV = Model(
    name="FermionQED-covariant",
    gauge_groups=(QED_GROUP,),
    fields=(PsiQEDField, GaugeField),
    lagrangian_decl=I * PsiQEDField.bar * Gamma(mu_decl) * CovD(PsiQEDField, mu_decl),
)

print("Declared fermion QED Lagrangian:", MODEL_QED_FERMION_COV.lagrangian_decl)
compiled_qed_cov = compile_covariant_terms(MODEL_QED_FERMION_COV)
print("Covariant fermion-QED terms:", labels(compiled_qed_cov))

LEGS_qed_decl = (
    PsiQEDField.leg(p1, conjugated=True, labels={SPINOR_KIND: i1}),
    PsiQEDField.leg(p2, labels={SPINOR_KIND: i2}),
    GaugeField.leg(p3, labels={LORENTZ_KIND: mu3}),
)

V_qed_cov = model_vertex(compiled_qed_cov[0], LEGS_qed_decl, simplify_gamma=True)
print("=====Dirac kinetic term (QED) via lagrangian_decl=====")
show("Vertex qbar q A from I * psibar * Gamma(mu) * CovD(psi, mu)", V_qed_cov)

MODEL_SCALAR_QED_COV = Model(
    name="ScalarQED-covariant",
    gauge_groups=(QED_GROUP,),
    fields=(PhiQEDField, GaugeField),
    lagrangian_decl=CovD(PhiQEDField.bar, mu_decl) * CovD(PhiQEDField, mu_decl),
)

print("Declared scalar QED Lagrangian:", MODEL_SCALAR_QED_COV.lagrangian_decl)
compiled_sqed = compile_covariant_terms(MODEL_SCALAR_QED_COV)
print("Covariant scalar-QED terms:", labels(compiled_sqed))

LEGS_sqed_current = (
    PhiQEDField.leg(p1, conjugated=True, species=b1),
    PhiQEDField.leg(p2, species=b2),
    GaugeField.leg(p3, labels={LORENTZ_KIND: mu3}, species=b3),
)
LEGS_sqed_contact = (
    PhiQEDField.leg(p1, conjugated=True, species=b1),
    PhiQEDField.leg(p2, species=b2),
    GaugeField.leg(p3, labels={LORENTZ_KIND: mu3}, species=b3),
    GaugeField.leg(p4, labels={LORENTZ_KIND: mu4}, species=b4),
)

sqed_species_3 = {b1: phiQEDdag0, b2: phiQED0, b3: A0}
sqed_species_4 = {b1: phiQEDdag0, b2: phiQED0, b3: A0, b4: A0}

V_sqed_current = sum(
    (model_vertex(t, LEGS_sqed_current, species_map=sqed_species_3) for t in compiled_sqed if "current" in t.label),
    Expression.num(0),
).expand()
V_sqed_contact = sum(
    (model_vertex(t, LEGS_sqed_contact, species_map=sqed_species_4) for t in compiled_sqed if "contact" in t.label),
    Expression.num(0),
).expand()

print("=====Complex-scalar kinetic term (QED) via lagrangian_decl — current=====")
show("Vertex phi^dagger phi A from CovD(phi.bar, mu) * CovD(phi, mu)", V_sqed_current)
print("=====Complex-scalar kinetic term (QED) via lagrangian_decl — contact=====")
show("Vertex phi^dagger phi A A from CovD(phi.bar, mu) * CovD(phi, mu)", V_sqed_contact)


Declared fermion QED Lagrangian: 1𝑖 * PsiQED.bar * Gamma(mu_decl) * CovD(PsiQED, mu_decl)
Covariant fermion-QED terms: ['i PsiQEDbar gamma^mu D_mu PsiQED', 'i PsiQEDbar gamma^mu D_mu PsiQED partial']
=====Dirac kinetic term (QED) via lagrangian_decl=====

=== Vertex qbar q A from I * psibar * Gamma(mu) * CovD(psi, mu) ===
-1𝑖*eQED*qPsi*gamma(bis(4, i1),bis(4, i2),mink(4, mu3))
Declared scalar QED Lagrangian: CovD(PhiQED.bar, mu_decl) * CovD(PhiQED, mu_decl)
Covariant scalar-QED terms: ['(D_mu PhiQED)^dagger (D^mu PhiQED) U1QED: scalar current (+)', '(D_mu PhiQED)^dagger (D^mu PhiQED) U1QED: scalar current (-)', '(D_mu PhiQED)^dagger (D^mu PhiQED) U1QED: scalar contact', '(D_mu PhiQED)^dagger (D^mu PhiQED) derivative']
=====Complex-scalar kinetic term (QED) via lagrangian_decl — current=====

=== Vertex phi^dagger phi A from CovD(phi.bar, mu) * CovD(phi, mu) ===
-1𝑖*eQED*qPhi*pcomp(p1,mu)+1𝑖*eQED*qPhi*pcomp(p2,mu)
=====Complex-scalar kinetic term (QED) via lagrangian_decl — contact=====

## 9.1 Covariant derivative compiler (matter kinetic terms)

In this codebase, `CovD(...)` means the gauge-theory covariant derivative.

### Conventions (frozen for the physical compiler)

- **Fourier-space derivative rule**: `partial_mu -> -i p_mu`
- **Overall vertex factor**: `vertex_factor(...)` contributes the universal `+i`
- **Matter covariant derivative**:

\[
D_\mu = \partial_\mu + i g A_\mu
\]

### What `compile_covariant_terms(...)` now lowers

The declarative front-end lowers the covered canonical forms:

- `I * Psi.bar * Gamma(mu) * CovD(Psi, mu)`
- `CovD(Phi.bar, mu) * CovD(Phi, mu)`

into ordinary `InteractionTerm` objects.

Those `InteractionTerm`s are still evaluated by the same `vertex_factor(...)`
engine as any other interaction.

### Repeated same-kind index slots and `slot_policy`

If a field carries the same index type more than once, gauge generators can be
ambiguous.

Rule:

- ambiguity -> error by default (`slot_policy="unique"`)
- explicit opt-in for tensor-product semantics (`slot_policy="sum"`)


In [45]:
# --- Fermion QCD: I * qbar * Gamma(mu) * CovD(q, mu) ---
mu_qcd_decl = S("mu_qcd_decl")
MODEL_QCD_FERMION_COV = Model(
    name="QCD-covariant",
    gauge_groups=(QCD_GROUP,),
    fields=(QuarkField, GluonField),
    lagrangian_decl=I * QuarkField.bar * Gamma(mu_qcd_decl) * CovD(QuarkField, mu_qcd_decl),
)

compiled_qcd_cov = compile_covariant_terms(MODEL_QCD_FERMION_COV)
print("Declared QCD fermion Lagrangian:", MODEL_QCD_FERMION_COV.lagrangian_decl)
print("Generated terms:", labels(compiled_qcd_cov))

LEGS_qcd_decl = (
    QuarkField.leg(p1, conjugated=True, species=b1, labels={SPINOR_KIND: i1, COLOR_FUND_KIND: c1}),
    QuarkField.leg(p2, species=b2, labels={SPINOR_KIND: i2, COLOR_FUND_KIND: c2}),
    GluonField.leg(p3, species=b3, labels={LORENTZ_KIND: mu3, COLOR_ADJ_KIND: a3}),
)

V_qcd_cov = model_vertex(compiled_qcd_cov[0], LEGS_qcd_decl, species_map={b1: psibar0, b2: psi0, b3: G0}, simplify_gamma=True)
show("QCD quark-gluon vertex from I * qbar * Gamma(mu) * CovD(q, mu)", V_qcd_cov)


Declared QCD fermion Lagrangian: 1𝑖 * q.bar * Gamma(mu_qcd_decl) * CovD(q, mu_qcd_decl)
Generated terms: ['i qbar gamma^mu D_mu q [ColorFund]', 'i qbar gamma^mu D_mu q partial']

=== QCD quark-gluon vertex from I * qbar * Gamma(mu) * CovD(q, mu) ===
-1𝑖*gS*gamma(bis(4, i1),bis(4, i2),mink(4, mu3))*t(coad(8, a3),cof(3, c1),cof(3, c2))


In [46]:
# --- Scalar QCD: CovD(PhiQCD.bar, mu) * CovD(PhiQCD, mu) ---
mu_sqcd_decl = S("mu_sqcd_decl")
MODEL_SCALAR_QCD_COV = Model(
    name="ScalarQCD-covariant",
    gauge_groups=(QCD_GROUP,),
    fields=(PhiQCDField, GluonField),
    lagrangian_decl=CovD(PhiQCDField.bar, mu_sqcd_decl) * CovD(PhiQCDField, mu_sqcd_decl),
)

compiled_sqcd = compile_covariant_terms(MODEL_SCALAR_QCD_COV)
print("Declared scalar QCD Lagrangian:", MODEL_SCALAR_QCD_COV.lagrangian_decl)
print("Generated terms:", labels(compiled_sqcd))

LEGS_sqcd_current = (
    PhiQCDField.leg(p1, conjugated=True, species=b1, labels={COLOR_FUND_KIND: c1}),
    PhiQCDField.leg(p2, species=b2, labels={COLOR_FUND_KIND: c2}),
    GluonField.leg(p3, species=b3, labels={LORENTZ_KIND: mu3, COLOR_ADJ_KIND: a3}),
)
LEGS_sqcd_contact = (
    PhiQCDField.leg(p1, conjugated=True, species=b1, labels={COLOR_FUND_KIND: c1}),
    PhiQCDField.leg(p2, species=b2, labels={COLOR_FUND_KIND: c2}),
    GluonField.leg(p3, species=b3, labels={LORENTZ_KIND: mu3, COLOR_ADJ_KIND: a3}),
    GluonField.leg(p4, species=b4, labels={LORENTZ_KIND: mu4, COLOR_ADJ_KIND: a4}),
)

sqcd_species_3 = {b1: phiQCDdag0, b2: phiQCD0, b3: G0}
sqcd_species_4 = {b1: phiQCDdag0, b2: phiQCD0, b3: G0, b4: G0}

V_sqcd_current = sum(
    (model_vertex(t, LEGS_sqcd_current, species_map=sqcd_species_3) for t in compiled_sqcd if "current" in t.label),
    Expression.num(0),
).expand()
V_sqcd_contact = sum(
    (model_vertex(t, LEGS_sqcd_contact, species_map=sqcd_species_4) for t in compiled_sqcd if "contact" in t.label),
    Expression.num(0),
).expand()

show("Scalar QCD current from CovD(Phi.bar, mu) * CovD(Phi, mu)", V_sqcd_current)
show("Scalar QCD contact from CovD(Phi.bar, mu) * CovD(Phi, mu)", V_sqcd_contact)


Declared scalar QCD Lagrangian: CovD(PhiQCD.bar, mu_sqcd_decl) * CovD(PhiQCD, mu_sqcd_decl)
Generated terms: ['(D_mu PhiQCD)^dagger (D^mu PhiQCD) SU3C: scalar current (+)', '(D_mu PhiQCD)^dagger (D^mu PhiQCD) SU3C: scalar current (-)', '(D_mu PhiQCD)^dagger (D^mu PhiQCD) SU3C: scalar contact [slots 1,1]', '(D_mu PhiQCD)^dagger (D^mu PhiQCD) derivative']

=== Scalar QCD current from CovD(Phi.bar, mu) * CovD(Phi, mu) ===
-1𝑖*gS*t(coad(8, a3),cof(3, c1),cof(3, c2))*pcomp(p1,mu)+1𝑖*gS*t(coad(8, a3),cof(3, c1),cof(3, c2))*pcomp(p2,mu)

=== Scalar QCD contact from CovD(Phi.bar, mu) * CovD(Phi, mu) ===
1𝑖*gS^2*g(mink(4, mu3),mink(4, mu4))*t(coad(8, a3),cof(3, c1),cof(3, c_mid_PhiQCD_SU3C))*t(coad(8, a4),cof(3, c_mid_PhiQCD_SU3C),cof(3, c2))+1𝑖*gS^2*g(mink(4, mu3),mink(4, mu4))*t(coad(8, a3),cof(3, c_mid_PhiQCD_SU3C),cof(3, c2))*t(coad(8, a4),cof(3, c1),cof(3, c_mid_PhiQCD_SU3C))


In [47]:
# --- Mixed QCD + QED fermion kinetic term expands into separate gauge pieces ---
qMix = S("qMix")
PsiMixField = Field(
    "PsiMix",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=S("psiMix0"),
    conjugate_symbol=S("psibarMix0"),
    indices=(SPINOR_INDEX, COLOR_FUND_INDEX),
    quantum_numbers={"Q": qMix},
)

mu_mix_decl = S("mu_mix_decl")
MODEL_MIXED_FERMION_COV = Model(
    name="MixedFermionQCDQED-covariant",
    gauge_groups=(QCD_GROUP, QED_GROUP),
    fields=(PsiMixField, GluonField, GaugeField),
    lagrangian_decl=I * PsiMixField.bar * Gamma(mu_mix_decl) * CovD(PsiMixField, mu_mix_decl),
)

compiled_mixed = compile_covariant_terms(MODEL_MIXED_FERMION_COV)
print("Declared mixed fermion Lagrangian:", MODEL_MIXED_FERMION_COV.lagrangian_decl)
print("Generated terms:")
for lab in labels(compiled_mixed):
    print(" -", lab)

LEGS_mixed_gluon = (
    PsiMixField.leg(p1, conjugated=True, species=b1, labels={SPINOR_KIND: i1, COLOR_FUND_KIND: c1}),
    PsiMixField.leg(p2, species=b2, labels={SPINOR_KIND: i2, COLOR_FUND_KIND: c2}),
    GluonField.leg(p3, species=b3, labels={LORENTZ_KIND: mu3, COLOR_ADJ_KIND: a3}),
)
LEGS_mixed_photon = (
    PsiMixField.leg(p1, conjugated=True, species=b1, labels={SPINOR_KIND: i1, COLOR_FUND_KIND: c1}),
    PsiMixField.leg(p2, species=b2, labels={SPINOR_KIND: i2, COLOR_FUND_KIND: c2}),
    GaugeField.leg(p3, species=b3, labels={LORENTZ_KIND: mu3}),
)

mixed_species_gluon = {b1: S("psibarMix0"), b2: S("psiMix0"), b3: G0}
mixed_species_photon = {b1: S("psibarMix0"), b2: S("psiMix0"), b3: A0}

# For the mixed fermion case the current labels do not carry the gauge-group name,
# so we use the generated term order: non-abelian piece first, abelian piece second.
V_mixed_gluon = model_vertex(compiled_mixed[0], LEGS_mixed_gluon, species_map=mixed_species_gluon, simplify_gamma=True)
V_mixed_photon = model_vertex(compiled_mixed[1], LEGS_mixed_photon, species_map=mixed_species_photon, simplify_gamma=True)

show("Mixed fermion gluon vertex", V_mixed_gluon)
show("Mixed fermion photon vertex", V_mixed_photon)


Declared mixed fermion Lagrangian: 1𝑖 * PsiMix.bar * Gamma(mu_mix_decl) * CovD(PsiMix, mu_mix_decl)
Generated terms:
 - i PsiMixbar gamma^mu D_mu PsiMix [ColorFund]
 - i PsiMixbar gamma^mu D_mu PsiMix
 - i PsiMixbar gamma^mu D_mu PsiMix partial

=== Mixed fermion gluon vertex ===
-1𝑖*gS*gamma(bis(4, i1),bis(4, i2),mink(4, mu3))*t(coad(8, a3),cof(3, c1),cof(3, c2))

=== Mixed fermion photon vertex ===
-1𝑖*eQED*qMix*g(cof(3, c1),cof(3, c2))*gamma(bis(4, i1),bis(4, i2),mink(4, mu3))


### Mixed-group complex scalar kinetic term

The scalar analogue of the mixed QCD+QED fermion example uses the same
`lagrangian_decl` entry point.

For one complex scalar carrying both an SU(3) representation and a U(1) charge,
`CovD(Phi.bar, mu) * CovD(Phi, mu)` generates:

- the SU(3) current
- the U(1) current
- the mixed SU(3) x U(1) contact term


In [48]:
# --- Mixed QCD + QED complex scalar kinetic term: currents + mixed contact ---
phiMix0, phiMixdag0 = S("phiMix0", "phiMixdag0")
qPhiMix = S("qPhiMix")
PhiMixField = Field(
    "PhiMix",
    spin=0,
    self_conjugate=False,
    symbol=phiMix0,
    conjugate_symbol=phiMixdag0,
    indices=(COLOR_FUND_INDEX,),
    quantum_numbers={"Q": qPhiMix},
)
MODEL_MIXED_SCALAR_COV = Model(
    name="MixedScalarQCDQED-covariant",
    gauge_groups=(QCD_GROUP, QED_GROUP),
    fields=(PhiMixField, GluonField, GaugeField),
    lagrangian_decl=CovD(PhiMixField.bar, mu_decl) * CovD(PhiMixField, mu_decl),
)
compiled_mixed_scalar = compile_covariant_terms(MODEL_MIXED_SCALAR_COV)
print("Declared mixed scalar Lagrangian:", MODEL_MIXED_SCALAR_COV.lagrangian_decl)
print("Generated terms:")
for lab in labels(compiled_mixed_scalar):
    print(" -", lab)

mixed_scalar_qcd_terms = [t for t in compiled_mixed_scalar if "SU3C: scalar current" in t.label]
mixed_scalar_qed_terms = [t for t in compiled_mixed_scalar if "U1QED: scalar current" in t.label]
mixed_scalar_contact_terms = [t for t in compiled_mixed_scalar if "mixed contact" in t.label]

LEGS_mixed_scalar_gluon = (
    PhiMixField.leg(p1, conjugated=True, species=b1, labels={COLOR_FUND_KIND: c1}),
    PhiMixField.leg(p2, species=b2, labels={COLOR_FUND_KIND: c2}),
    GluonField.leg(p3, labels={LORENTZ_KIND: mu3, COLOR_ADJ_KIND: a3}, species=b3),
)
LEGS_mixed_scalar_photon = (
    PhiMixField.leg(p1, conjugated=True, species=b1, labels={COLOR_FUND_KIND: c1}),
    PhiMixField.leg(p2, species=b2, labels={COLOR_FUND_KIND: c2}),
    GaugeField.leg(p3, labels={LORENTZ_KIND: mu3}, species=b3),
)
LEGS_mixed_scalar_contact = (
    PhiMixField.leg(p1, conjugated=True, species=b1, labels={COLOR_FUND_KIND: c1}),
    PhiMixField.leg(p2, species=b2, labels={COLOR_FUND_KIND: c2}),
    GluonField.leg(p3, labels={LORENTZ_KIND: mu3, COLOR_ADJ_KIND: a3}, species=b3),
    GaugeField.leg(p4, labels={LORENTZ_KIND: mu4}, species=b4),
)

mixed_scalar_species_gluon = {b1: phiMixdag0, b2: phiMix0, b3: G0}
mixed_scalar_species_photon = {b1: phiMixdag0, b2: phiMix0, b3: A0}
mixed_scalar_species_contact = {b1: phiMixdag0, b2: phiMix0, b3: G0, b4: A0}

V_mixed_scalar_gluon = sum(
    (model_vertex(t, LEGS_mixed_scalar_gluon, species_map=mixed_scalar_species_gluon) for t in mixed_scalar_qcd_terms),
    Expression.num(0),
).expand()
V_mixed_scalar_photon = sum(
    (model_vertex(t, LEGS_mixed_scalar_photon, species_map=mixed_scalar_species_photon) for t in mixed_scalar_qed_terms),
    Expression.num(0),
).expand()
V_mixed_scalar_contact = sum(
    (model_vertex(t, LEGS_mixed_scalar_contact, species_map=mixed_scalar_species_contact) for t in mixed_scalar_contact_terms),
    Expression.num(0),
).expand()

show("Mixed scalar gluon current", V_mixed_scalar_gluon)
show("Mixed scalar photon current", V_mixed_scalar_photon)
show("Mixed scalar SU(3) x U(1) contact", V_mixed_scalar_contact)


Declared mixed scalar Lagrangian: CovD(PhiMix.bar, mu_decl) * CovD(PhiMix, mu_decl)
Generated terms:
 - (D_mu PhiMix)^dagger (D^mu PhiMix) SU3C: scalar current (+)
 - (D_mu PhiMix)^dagger (D^mu PhiMix) SU3C: scalar current (-)
 - (D_mu PhiMix)^dagger (D^mu PhiMix) SU3C: scalar contact [slots 1,1]
 - (D_mu PhiMix)^dagger (D^mu PhiMix) U1QED: scalar current (+)
 - (D_mu PhiMix)^dagger (D^mu PhiMix) U1QED: scalar current (-)
 - (D_mu PhiMix)^dagger (D^mu PhiMix) U1QED: scalar contact
 - (D_mu PhiMix)^dagger (D^mu PhiMix) SU3C x U1QED: scalar mixed contact [slot 1]
 - (D_mu PhiMix)^dagger (D^mu PhiMix) U1QED x SU3C: scalar mixed contact [slot 1]
 - (D_mu PhiMix)^dagger (D^mu PhiMix) derivative

=== Mixed scalar gluon current ===
-1𝑖*gS*t(coad(8, a3),cof(3, c1),cof(3, c2))*pcomp(p1,mu)+1𝑖*gS*t(coad(8, a3),cof(3, c1),cof(3, c2))*pcomp(p2,mu)

=== Mixed scalar photon current ===
-1𝑖*eQED*qPhiMix*g(cof(3, c1),cof(3, c2))*pcomp(p1,mu)+1𝑖*eQED*qPhiMix*g(cof(3, c1),cof(3, c2))*pcomp(p2,mu)

=== M

In [49]:
# --- Bislotted scalar with explicit opt-in: slot_policy='sum' ---
COLOR_FUND_REP_SUM = GaugeRepresentation(
    index=COLOR_FUND_INDEX,
    generator_builder=gauge_generator,
    name="fundamental_sum",
    slot_policy="sum",
)
QCD_GROUP_BISLOT_SUM = GaugeGroup(
    name="SU3CBiSum",
    abelian=False,
    coupling=gS,
    gauge_boson=G0,
    structure_constant=structure_constant,
    representations=(COLOR_FUND_REP_SUM,),
)
MODEL_BISLOT_SUM_COV = Model(
    name="ScalarQCD-bislot-covariant-sum",
    gauge_groups=(QCD_GROUP_BISLOT_SUM,),
    fields=(PhiBiField, GluonField),
    lagrangian_decl=CovD(PhiBiField.bar, S("mu_bis_decl")) * CovD(PhiBiField, S("mu_bis_decl")),
)
compiled_bislot_sum = compile_covariant_terms(MODEL_BISLOT_SUM_COV)
print("Bislotted scalar declaration:", MODEL_BISLOT_SUM_COV.lagrangian_decl)
print("Generated labels:")
for lab in labels(compiled_bislot_sum):
    print(" -", lab)

LEGS_bislot_current = (
    PhiBiField.leg(p1, conjugated=True, species=b1, labels={COLOR_FUND_KIND: (c1, c3)}),
    PhiBiField.leg(p2, species=b2, labels={COLOR_FUND_KIND: (c2, c4)}),
    GluonField.leg(p3, labels={LORENTZ_KIND: mu3, COLOR_ADJ_KIND: a3}, species=b3),
)
LEGS_bislot_contact = (
    PhiBiField.leg(p1, conjugated=True, species=b1, labels={COLOR_FUND_KIND: (c1, c3)}),
    PhiBiField.leg(p2, species=b2, labels={COLOR_FUND_KIND: (c2, c4)}),
    GluonField.leg(p3, labels={LORENTZ_KIND: mu3, COLOR_ADJ_KIND: a3}, species=b3),
    GluonField.leg(p4, labels={LORENTZ_KIND: mu4, COLOR_ADJ_KIND: a4}, species=b4),
)

bislot_species_3 = {b1: phiBidag0, b2: phiBi0, b3: G0}
bislot_species_4 = {b1: phiBidag0, b2: phiBi0, b3: G0, b4: G0}

V_bislot_current_sum = sum(
    (model_vertex(t, LEGS_bislot_current, species_map=bislot_species_3) for t in compiled_bislot_sum if "current" in t.label),
    Expression.num(0),
).expand()
V_bislot_contact_sum = sum(
    (model_vertex(t, LEGS_bislot_contact, species_map=bislot_species_4) for t in compiled_bislot_sum if "contact" in t.label),
    Expression.num(0),
).expand()

show("Bislot scalar summed current", V_bislot_current_sum)
show("Bislot scalar summed contact", V_bislot_contact_sum)


Bislotted scalar declaration: CovD(PhiBi.bar, mu_bis_decl) * CovD(PhiBi, mu_bis_decl)
Generated labels:
 - (D_mu PhiBi)^dagger (D^mu PhiBi) SU3CBiSum: scalar current (+)_slot1
 - (D_mu PhiBi)^dagger (D^mu PhiBi) SU3CBiSum: scalar current (-)_slot1
 - (D_mu PhiBi)^dagger (D^mu PhiBi) SU3CBiSum: scalar current (+)_slot2
 - (D_mu PhiBi)^dagger (D^mu PhiBi) SU3CBiSum: scalar current (-)_slot2
 - (D_mu PhiBi)^dagger (D^mu PhiBi) SU3CBiSum: scalar contact [slots 1,1]
 - (D_mu PhiBi)^dagger (D^mu PhiBi) SU3CBiSum: scalar contact [slots 1,2]
 - (D_mu PhiBi)^dagger (D^mu PhiBi) SU3CBiSum: scalar contact [slots 2,1]
 - (D_mu PhiBi)^dagger (D^mu PhiBi) SU3CBiSum: scalar contact [slots 2,2]
 - (D_mu PhiBi)^dagger (D^mu PhiBi) derivative

=== Bislot scalar summed current ===
-1𝑖*gS*g(cof(3, c1),cof(3, c2))*t(coad(8, a3),cof(3, c3),cof(3, c4))*pcomp(p1,mu)+1𝑖*gS*g(cof(3, c1),cof(3, c2))*t(coad(8, a3),cof(3, c3),cof(3, c4))*pcomp(p2,mu)-1𝑖*gS*g(cof(3, c3),cof(3, c4))*t(coad(8, a3),cof(3, c1),cof(3, c

### Unbroken electroweak slice

A minimal unbroken `SU(2)_L x U(1)_Y` matter layer already works through
the same `lagrangian_decl` front end.

Keep the picture small:

- one fermion doublet `L`
- one Higgs-like complex scalar doublet `H`
- the unbroken gauge bosons `W` and `B`

From those declarations we can ask directly for:

- the fermion current `Lbar L W`
- the Higgs hypercharge current `Hdag H B`
- the mixed electroweak contact term `Hdag H W B`

There is still no symmetry breaking here. This is only the unbroken
electroweak matter layer.


In [50]:

mu = S("mu")
g1 = S("g1")
g2 = S("g2")
yL = S("yL")
yH = S("yH")

WField = Field(
    "W",
    spin=1,
    self_conjugate=True,
    symbol=S("W0"),
    indices=(LORENTZ_INDEX, WEAK_ADJ_INDEX),
)

BField = Field(
    "B",
    spin=1,
    self_conjugate=True,
    symbol=S("B0"),
    indices=(LORENTZ_INDEX,),
)

LDoublet = Field(
    "L",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=S("L0"),
    conjugate_symbol=S("Lbar0"),
    indices=(SPINOR_INDEX, WEAK_FUND_INDEX),
    quantum_numbers={"Y": yL},
)

HDoublet = Field(
    "H",
    spin=0,
    self_conjugate=False,
    symbol=S("H0"),
    conjugate_symbol=S("Hdag0"),
    indices=(WEAK_FUND_INDEX,),
    quantum_numbers={"Y": yH},
)

WEAK_DOUBLET_REP = GaugeRepresentation(
    index=WEAK_FUND_INDEX,
    generator_builder=weak_gauge_generator,
    name="doublet",
)

SU2L = GaugeGroup(
    name="SU2L",
    abelian=False,
    coupling=g2,
    gauge_boson="W",
    structure_constant=weak_structure_constant,
    representations=(WEAK_DOUBLET_REP,),
)

U1Y = GaugeGroup(
    name="U1Y",
    abelian=True,
    coupling=g1,
    gauge_boson="B",
    charge="Y",
)

fermion_model = Model(
    name="EW-fermion-demo",
    gauge_groups=(SU2L, U1Y),
    fields=(LDoublet, WField, BField),
    lagrangian_decl=I * LDoublet.bar * Gamma(mu) * CovD(LDoublet, mu),
)

higgs_model = Model(
    name="EW-higgs-demo",
    gauge_groups=(SU2L, U1Y),
    fields=(HDoublet, WField, BField),
    lagrangian_decl=CovD(HDoublet.bar, mu) * CovD(HDoublet, mu),
)

print("Unbroken electroweak fermion current Γ(Lbar, L, W):")
print(fermion_model.lagrangian().feynman_rule(LDoublet.bar, LDoublet, WField))
print()

print("Unbroken electroweak Higgs current Γ(Hdag, H, B):")
print(higgs_model.lagrangian().feynman_rule(HDoublet.bar, HDoublet, BField))
print()

print("Unbroken electroweak mixed contact Γ(Hdag, H, W, B):")
print(higgs_model.lagrangian().feynman_rule(HDoublet.bar, HDoublet, WField, BField))

Unbroken electroweak fermion current Γ(Lbar, L, W):
-1𝑖*g2*(2*𝜋)^d*gamma(bis(4, i1),bis(4, i3),mink(4, i5))*t(coad(3, i6),cof(2, i2),cof(2, i4))*Delta(q1+q2+q3)

Unbroken electroweak Higgs current Γ(Hdag, H, B):
-1𝑖*g1*yH*(2*𝜋)^d*g(cof(2, i1),cof(2, i2))*pcomp(q1,mu)*Delta(q1+q2+q3)+1𝑖*g1*yH*(2*𝜋)^d*g(cof(2, i1),cof(2, i2))*pcomp(q2,mu)*Delta(q1+q2+q3)

Unbroken electroweak mixed contact Γ(Hdag, H, W, B):
2𝑖*g1*g2*yH*(2*𝜋)^d*g(mink(4, i3),mink(4, i5))*t(coad(3, i4),cof(2, i1),cof(2, i2))*Delta(q1+q2+q3+q4)


## 10. Pure-gauge sector and Yang-Mills self-interactions

Pure-gauge interactions are now declared through the same front-end:

- `-(1/4) * FieldStrength(G, mu, nu) * FieldStrength(G, mu, nu)`

The compiler then generates:

- the gauge bilinear
- the Yang-Mills cubic vertex
- the Yang-Mills quartic vertex

for the covered non-abelian case.


In [51]:
mu_ym_decl, nu_ym_decl = S("mu_ym_decl", "nu_ym_decl")
MODEL_QCD_GAUGE_COV = Model(
    name="QCDGauge-covariant",
    gauge_groups=(QCD_GROUP,),
    fields=(GluonField,),
    lagrangian_decl=-(Expression.num(1) / Expression.num(4)) * FieldStrength(QCD_GROUP, mu_ym_decl, nu_ym_decl) * FieldStrength(QCD_GROUP, mu_ym_decl, nu_ym_decl),
)

print("Declared Yang-Mills Lagrangian:", MODEL_QCD_GAUGE_COV.lagrangian_decl)
compiled_ym = compile_covariant_terms(MODEL_QCD_GAUGE_COV)
print("Number of generated pure-gauge interactions:", len(compiled_ym))
print("Generated labels:")
for label in labels(compiled_ym):
    print(" -", label)

LEGS_two_gluon = (
    GluonField.leg(p1, species=b1, labels={LORENTZ_KIND: mu, COLOR_ADJ_KIND: a3}),
    GluonField.leg(p2, species=b2, labels={LORENTZ_KIND: nu, COLOR_ADJ_KIND: a4}),
)
LEGS_three_gluon = (
    GluonField.leg(p1, species=b1, labels={LORENTZ_KIND: mu, COLOR_ADJ_KIND: a3}),
    GluonField.leg(p2, species=b2, labels={LORENTZ_KIND: nu, COLOR_ADJ_KIND: a4}),
    GluonField.leg(p3, species=b3, labels={LORENTZ_KIND: rho, COLOR_ADJ_KIND: a5}),
)
LEGS_four_gluon = (
    GluonField.leg(p1, species=b1, labels={LORENTZ_KIND: mu, COLOR_ADJ_KIND: a3}),
    GluonField.leg(p2, species=b2, labels={LORENTZ_KIND: nu, COLOR_ADJ_KIND: a4}),
    GluonField.leg(p3, species=b3, labels={LORENTZ_KIND: rho, COLOR_ADJ_KIND: a5}),
    GluonField.leg(p4, species=b4, labels={LORENTZ_KIND: sigma, COLOR_ADJ_KIND: a6}),
)

V_ym_bilinear = sum(
    (model_vertex(t, LEGS_two_gluon, species_map={b1: G0, b2: G0}) for t in compiled_ym[:2]),
    Expression.num(0),
).expand()
V_ym_cubic = model_vertex(compiled_ym[2], LEGS_three_gluon, species_map={b1: G0, b2: G0, b3: G0})
V_ym_4 = model_vertex(compiled_ym[3], LEGS_four_gluon, species_map={b1: G0, b2: G0, b3: G0, b4: G0})

show("Yang-Mills bilinear from FieldStrength", V_ym_bilinear)
show("Yang-Mills cubic from FieldStrength", V_ym_cubic)
show("Yang-Mills quartic from FieldStrength", V_ym_4)


Declared Yang-Mills Lagrangian: -1/4 * FieldStrength(SU3C, mu_ym_decl, nu_ym_decl) * FieldStrength(SU3C, mu_ym_decl, nu_ym_decl)
Number of generated pure-gauge interactions: 4
Generated labels:
 - -1/4 SU3C field strength squared SU3C: gauge kinetic bilinear (metric)
 - -1/4 SU3C field strength squared SU3C: gauge kinetic bilinear (cross)
 - -1/4 SU3C field strength squared SU3C: Yang-Mills cubic
 - -1/4 SU3C field strength squared SU3C: Yang-Mills quartic

=== Yang-Mills bilinear from FieldStrength ===
1𝑖*g(mink(4, mu),mink(4, nu))*g(coad(8, a3),coad(8, a4))*pcomp(p1,rho_G_SU3C)*pcomp(p2,rho_G_SU3C)-1𝑖/2*g(coad(8, a3),coad(8, a4))*g(mink(4, mu),mink(4, rho_left_G_SU3C))*g(mink(4, nu),mink(4, rho_right_G_SU3C))*pcomp(p1,rho_right_G_SU3C)*pcomp(p2,rho_left_G_SU3C)-1𝑖/2*g(coad(8, a3),coad(8, a4))*g(mink(4, mu),mink(4, rho_right_G_SU3C))*g(mink(4, nu),mink(4, rho_left_G_SU3C))*pcomp(p1,rho_left_G_SU3C)*pcomp(p2,rho_right_G_SU3C)

=== Yang-Mills cubic from FieldStrength ===
1𝑖*(-1𝑖*gS*g(mi

## 10.5 Ordinary Gauge Fixing And Ghosts

Gauge fixing and ghosts now use typed declarative wrappers too:

- `GaugeFixing(G, xi=...)`
- `GhostLagrangian(G)`

These wrappers stay visible in the source declaration and lower onto the same
backend as the older helper classes.


In [52]:
from lagrangian.operators import gauge_fixing_bilinear, gauge_kinetic_bilinear, ghost_gauge, ghost_kinetic

xiQED, xiQCD = S("xiQED", "xiQCD")
ghG0, ghGbar0 = S("ghG0", "ghGbar0")

GhostGluonField = Field(
    "ghG",
    spin=0,
    kind="ghost",
    self_conjugate=False,
    symbol=ghG0,
    conjugate_symbol=ghGbar0,
    indices=(COLOR_ADJ_INDEX,),
)

QCD_GROUP_GHOST = GaugeGroup(
    name="SU3C",
    abelian=False,
    coupling=gS,
    gauge_boson=G0,
    ghost_field=ghG0,
    structure_constant=structure_constant,
    representations=(COLOR_FUND_REP,),
)

MODEL_QED_GAUGE_FIXING = Model(
    name="QEDGaugeFixing-covariant",
    gauge_groups=(QED_GROUP,),
    fields=(GaugeField,),
    lagrangian_decl=GaugeFixing(QED_GROUP, xi=xiQED),
)
compiled_qed_gf = compile_covariant_terms(MODEL_QED_GAUGE_FIXING)
print("QED gauge-fixing declaration:", MODEL_QED_GAUGE_FIXING.lagrangian_decl)
print("QED gauge-fixing terms:", labels(compiled_qed_gf))

LEGS_qed_gf = (
    GaugeField.leg(p1, species=b1, labels={LORENTZ_KIND: mu3}),
    GaugeField.leg(p2, species=b2, labels={LORENTZ_KIND: mu4}),
)
V_qed_gf = model_vertex(compiled_qed_gf[0], LEGS_qed_gf, species_map={b1: A0, b2: A0})
show("Gauge fixing (QED)", V_qed_gf)
show("Compact form", (I / xiQED) * gauge_fixing_bilinear(mu3, mu4, p1, p2))

MODEL_QCD_GAUGE_FIXING = Model(
    name="QCDGaugeFixing-covariant",
    gauge_groups=(QCD_GROUP,),
    fields=(GluonField,),
    lagrangian_decl=GaugeFixing(QCD_GROUP, xi=xiQCD),
)
compiled_qcd_gf = compile_covariant_terms(MODEL_QCD_GAUGE_FIXING)
print("QCD gauge-fixing declaration:", MODEL_QCD_GAUGE_FIXING.lagrangian_decl)
print("QCD gauge-fixing terms:", labels(compiled_qcd_gf))

LEGS_qcd_gf = (
    GluonField.leg(p1, species=b1, labels={LORENTZ_KIND: mu3, COLOR_ADJ_KIND: a3}),
    GluonField.leg(p2, species=b2, labels={LORENTZ_KIND: mu4, COLOR_ADJ_KIND: a4}),
)
V_qcd_gf = model_vertex(compiled_qcd_gf[0], LEGS_qcd_gf, species_map={b1: G0, b2: G0})
show("Gauge fixing (QCD)", V_qcd_gf)
show("Compact form", (I / xiQCD) * gauge_fixing_bilinear(mu3, mu4, p1, p2) * COLOR_ADJ_INDEX.representation.g(a3, a4).to_expression())

MODEL_QED_GAUGE_FIXED = Model(
    name="QEDGaugeFixed-covariant",
    gauge_groups=(QED_GROUP,),
    fields=(GaugeField,),
    lagrangian_decl=-(Expression.num(1) / Expression.num(4)) * FieldStrength(QED_GROUP, mu, nu) * FieldStrength(QED_GROUP, mu, nu) + GaugeFixing(QED_GROUP, xi=xiQED),
)
compiled_qed_gauge_fixed = compile_covariant_terms(MODEL_QED_GAUGE_FIXED)
V_qed_gauge_fixed = simplify_gamma_chain(
    sum((model_vertex(t, LEGS_qed_gf, species_map={b1: A0, b2: A0}) for t in compiled_qed_gauge_fixed), Expression.num(0))
).expand()
photon_rho = compiled_qed_gauge_fixed[0].derivatives[0].lorentz_index
show("Ordinary gauge-fixed photon bilinear", V_qed_gauge_fixed)
show("Compact form", I * (gauge_kinetic_bilinear(mu3, mu4, p1, p2, photon_rho) + gauge_fixing_bilinear(mu3, mu4, p1, p2) / xiQED))

MODEL_QCD_GHOST = Model(
    name="QCDGhost-covariant",
    gauge_groups=(QCD_GROUP_GHOST,),
    fields=(GluonField, GhostGluonField),
    lagrangian_decl=GhostLagrangian(QCD_GROUP_GHOST),
)
compiled_qcd_ghost = compile_covariant_terms(MODEL_QCD_GHOST)
print("QCD ghost declaration:", MODEL_QCD_GHOST.lagrangian_decl)
print("QCD ghost terms:", labels(compiled_qcd_ghost))

LEGS_ghost_bilinear = (
    GhostGluonField.leg(p1, conjugated=True, species=b1, labels={COLOR_ADJ_KIND: a3}),
    GhostGluonField.leg(p2, species=b2, labels={COLOR_ADJ_KIND: a4}),
)
LEGS_ghost_gluon = (
    GhostGluonField.leg(p1, conjugated=True, species=b1, labels={COLOR_ADJ_KIND: a3}),
    GluonField.leg(p2, species=b2, labels={LORENTZ_KIND: mu3, COLOR_ADJ_KIND: a4}),
    GhostGluonField.leg(p3, species=b3, labels={COLOR_ADJ_KIND: a5}),
)
V_ghost_bilinear = model_vertex(compiled_qcd_ghost[0], LEGS_ghost_bilinear, species_map={b1: ghGbar0, b2: ghG0})
V_ghost_gluon = model_vertex(compiled_qcd_ghost[1], LEGS_ghost_gluon, species_map={b1: ghGbar0, b2: G0, b3: ghG0})
show("Ghost bilinear", V_ghost_bilinear)
show("Compact form", -I * ghost_kinetic(a3, a4, p1, p2, S("rho_ghost")))
show("Ghost-gluon interaction", V_ghost_gluon)
show("Compact form", -gS * ghost_gauge(a3, a4, a5, mu3, p1))

MODEL_QCD_GAUGE_FIXED = Model(
    name="QCDGaugeFixed-covariant",
    gauge_groups=(QCD_GROUP_GHOST,),
    fields=(GluonField, GhostGluonField),
    lagrangian_decl=-(Expression.num(1) / Expression.num(4)) * FieldStrength(QCD_GROUP_GHOST, mu, nu) * FieldStrength(QCD_GROUP_GHOST, mu, nu) + GaugeFixing(QCD_GROUP_GHOST, xi=xiQCD) + GhostLagrangian(QCD_GROUP_GHOST),
)
compiled_qcd_gauge_fixed = compile_covariant_terms(MODEL_QCD_GAUGE_FIXED)
V_qcd_gauge_fixed = simplify_gamma_chain(
    model_vertex(compiled_qcd_gauge_fixed[0], LEGS_qcd_gf, species_map={b1: G0, b2: G0})
    + model_vertex(compiled_qcd_gauge_fixed[1], LEGS_qcd_gf, species_map={b1: G0, b2: G0})
    + model_vertex(compiled_qcd_gauge_fixed[4], LEGS_qcd_gf, species_map={b1: G0, b2: G0})
).expand()
gluon_rho = compiled_qcd_gauge_fixed[0].derivatives[0].lorentz_index
show("Ordinary gauge-fixed gluon bilinear", V_qcd_gauge_fixed)
show("Compact form", I * (gauge_kinetic_bilinear(mu3, mu4, p1, p2, gluon_rho) + gauge_fixing_bilinear(mu3, mu4, p1, p2) / xiQCD) * COLOR_ADJ_INDEX.representation.g(a3, a4).to_expression())


QED gauge-fixing declaration: GaugeFixing(U1QED, xi=xiQED)
QED gauge-fixing terms: ['-(1/2 xiQED) (U1QED gauge fixing)']

=== Gauge fixing (QED) ===
1𝑖*(1/2*g(mink(4, mu3),mink(4, rho_left_A_U1QED_gf))*g(mink(4, mu4),mink(4, rho_right_A_U1QED_gf))*pcomp(p1,rho_left_A_U1QED_gf)*pcomp(p2,rho_right_A_U1QED_gf)/xiQED+1/2*g(mink(4, mu3),mink(4, rho_right_A_U1QED_gf))*g(mink(4, mu4),mink(4, rho_left_A_U1QED_gf))*pcomp(p1,rho_right_A_U1QED_gf)*pcomp(p2,rho_left_A_U1QED_gf)/xiQED)

=== Compact form ===
1𝑖*pcomp(p1,mu4)*pcomp(p2,mu3)/xiQED
QCD gauge-fixing declaration: GaugeFixing(SU3C, xi=xiQCD)
QCD gauge-fixing terms: ['-(1/2 xiQCD) (SU3C gauge fixing)']

=== Gauge fixing (QCD) ===
1𝑖*(1/2*g(coad(8, a3),coad(8, a4))*g(mink(4, mu3),mink(4, rho_left_G_SU3C_gf))*g(mink(4, mu4),mink(4, rho_right_G_SU3C_gf))*pcomp(p1,rho_left_G_SU3C_gf)*pcomp(p2,rho_right_G_SU3C_gf)/xiQCD+1/2*g(coad(8, a3),coad(8, a4))*g(mink(4, mu3),mink(4, rho_right_G_SU3C_gf))*g(mink(4, mu4),mink(4, rho_left_G_SU3C_gf))*pcomp(p

## 11. Recommended `lagrangian_decl` API

Use `lagrangian_decl` as a sum of source terms.

1. define `Field`, `GaugeGroup`, and optional `GaugeRepresentation` metadata
2. write each source term directly in the front-end syntax
3. call `Model(...).lagrangian()` to compile the declaration
4. extract the vertex you want with `feynman_rule(...)`

Supported front-end building blocks:

- local monomials such as `lam * Phi * Phi * Phi * Phi`
- local derivative monomials through `PartialD(...)`, for example `PartialD(Phi, mu) * PartialD(Phi, mu) * Phi * Phi`
- local fermion derivative operators such as `Psi.bar * Gamma(mu) * PartialD(Psi, nu) * Phi * Chi`
- fermions: `I * Psi.bar * Gamma(mu) * CovD(Psi, mu)`
- complex scalars: `CovD(Phi.bar, mu) * CovD(Phi, mu)`
- dressed covariant terms such as `I * Psi.bar * Gamma(mu) * CovD(Psi, mu) * Phi * Phi`
- pure gauge: `-(1/4) * FieldStrength(G, mu, nu) * FieldStrength(G, mu, nu)`
- gauge fixing: `GaugeFixing(G, xi=...)`
- ghosts: `GhostLagrangian(G)`

Internally, the declaration is compiled one source term at a time. Purely local
terms lower directly to ordinary interactions, while only terms containing
canonical `CovD`, `FieldStrength`, gauge-fixing, or ghost structure are expanded.

`InteractionTerm(...)` still exists as a backend escape hatch for genuinely exotic
local tensor structures, but it is no longer the normal front-end path.


In [53]:
# Scalar: a generic local operator can be declared directly as a field product.
Phi = Field("Phi", spin=0, self_conjugate=True, symbol=S("phi"))
lam_phi4 = S("lam_phi4")
MODEL_SCALAR4 = Model(fields=(Phi,), lagrangian_decl=lam_phi4*Phi*Phi*Phi*Phi)
L_scalar = MODEL_SCALAR4.lagrangian()

print("Source declaration:", MODEL_SCALAR4.lagrangian_decl)
show("Scalar vertex phi phi phi phi", L_scalar.feynman_rule(Phi, Phi, Phi, Phi))


Source declaration: lam_phi4 * Phi * Phi * Phi * Phi

=== Scalar vertex phi phi phi phi ===
24𝑖*lam_phi4*(2*𝜋)^d*Delta(q1+q2+q3+q4)


In [54]:
# Several source terms can be summed in one declaration; each term is compiled separately.
Phi = Field("Phi", spin=0, self_conjugate=True, symbol=S("phi"))
m2 = S("m2")
lam_phi4 = S("lam_phi4")
MODEL_SUM = Model(
    fields=(Phi,),
    lagrangian_decl=-m2 * Phi * Phi + lam_phi4 * Phi * Phi * Phi * Phi,
)
L_sum = MODEL_SUM.lagrangian()

print("Source declaration:", MODEL_SUM.lagrangian_decl)
show("Mass vertex phi phi", L_sum.feynman_rule(Phi, Phi))
show("Quartic vertex phi phi phi phi", L_sum.feynman_rule(Phi, Phi, Phi, Phi))


Source declaration: -m2 * Phi * Phi + lam_phi4 * Phi * Phi * Phi * Phi

=== Mass vertex phi phi ===
-2𝑖*m2*(2*𝜋)^d*Delta(q1+q2)

=== Quartic vertex phi phi phi phi ===
24𝑖*lam_phi4*(2*𝜋)^d*Delta(q1+q2+q3+q4)


In [55]:
# Fermion: the canonical covariant kinetic term generates the gauge current.
mu = S("mu")
Psi = Field(
    "Psi",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=S("psi"),
    conjugate_symbol=S("psibar"),
    indices=(SPINOR_INDEX,),
    quantum_numbers={"Q": S("Qpsi")},
)
A = Field("A", spin=1, self_conjugate=True, symbol=S("A"), indices=(LORENTZ_INDEX,))
U1 = GaugeGroup(name="U1", abelian=True, coupling=S("e"), gauge_boson=A.symbol, charge="Q")
MODEL_FERMION = Model(
    gauge_groups=(U1,),
    fields=(Psi, A),
    lagrangian_decl=I * Psi.bar * Gamma(mu) * CovD(Psi, mu),
)
L_fermion = MODEL_FERMION.lagrangian()

print("Source declaration:", MODEL_FERMION.lagrangian_decl)
show("Fermion vertex psibar psi A", L_fermion.feynman_rule(Psi.bar, Psi, A))


Source declaration: 1𝑖 * Psi.bar * Gamma(mu) * CovD(Psi, mu)

=== Fermion vertex psibar psi A ===
-1𝑖*Qpsi*e*(2*𝜋)^d*gamma(bis(4, i1),bis(4, i2),mink(4, i3))*Delta(q1+q2+q3)


In [56]:
# Complex scalar: the canonical covariant kinetic term generates both current and contact pieces.
mu = S("mu")
PhiC = Field(
    "PhiC",
    spin=0,
    self_conjugate=False,
    symbol=S("phiC"),
    conjugate_symbol=S("phiCbar"),
    quantum_numbers={"Q": S("Qphi")},
)
A = Field("A", spin=1, self_conjugate=True, symbol=S("A"), indices=(LORENTZ_INDEX,))
U1 = GaugeGroup(name="U1", abelian=True, coupling=S("e"), gauge_boson=A.symbol, charge="Q")
MODEL_SCALAR_GAUGE = Model(
    gauge_groups=(U1,),
    fields=(PhiC, A),
    lagrangian_decl=CovD(PhiC.bar, mu) * CovD(PhiC, mu),
)
L_scalar_gauge = MODEL_SCALAR_GAUGE.lagrangian()

print("Source declaration:", MODEL_SCALAR_GAUGE.lagrangian_decl)
show("Complex-scalar current phi^dag phi A", L_scalar_gauge.feynman_rule(PhiC.bar, PhiC, A))
show("Complex-scalar contact phi^dag phi A A", L_scalar_gauge.feynman_rule(PhiC.bar, PhiC, A, A))


Source declaration: CovD(PhiC.bar, mu) * CovD(PhiC, mu)

=== Complex-scalar current phi^dag phi A ===
-1𝑖*e*Qphi*(2*𝜋)^d*pcomp(q1,mu)*Delta(q1+q2+q3)+1𝑖*e*Qphi*(2*𝜋)^d*pcomp(q2,mu)*Delta(q1+q2+q3)

=== Complex-scalar contact phi^dag phi A A ===
2𝑖*e^2*Qphi^2*(2*𝜋)^d*g(mink(4, i1),mink(4, i2))*Delta(q1+q2+q3+q4)


In [57]:
# Derivatives: ordinary local derivative operators can be declared with PartialD(...).
Phi = Field("Phi", spin=0, self_conjugate=True, symbol=S("phi"))
mu = S("mu")
MODEL_DERIV = Model(
    fields=(Phi,),
    lagrangian_decl=S("gD2") * PartialD(Phi, mu) * PartialD(Phi, mu) * Phi * Phi,
)
L_deriv = MODEL_DERIV.lagrangian()

print("Source declaration:", MODEL_DERIV.lagrangian_decl)
show("Derivative vertex (d phi)^2 phi^2", L_deriv.feynman_rule(Phi, Phi, Phi, Phi))


Source declaration: gD2 * PartialD(Phi, mu) * PartialD(Phi, mu) * Phi * Phi

=== Derivative vertex (d phi)^2 phi^2 ===
-4𝑖*gD2*(2*𝜋)^d*pcomp(q1,mu)*pcomp(q2,mu)*Delta(q1+q2+q3+q4)-4𝑖*gD2*(2*𝜋)^d*pcomp(q1,mu)*pcomp(q3,mu)*Delta(q1+q2+q3+q4)-4𝑖*gD2*(2*𝜋)^d*pcomp(q1,mu)*pcomp(q4,mu)*Delta(q1+q2+q3+q4)-4𝑖*gD2*(2*𝜋)^d*pcomp(q2,mu)*pcomp(q3,mu)*Delta(q1+q2+q3+q4)-4𝑖*gD2*(2*𝜋)^d*pcomp(q2,mu)*pcomp(q4,mu)*Delta(q1+q2+q3+q4)-4𝑖*gD2*(2*𝜋)^d*pcomp(q3,mu)*pcomp(q4,mu)*Delta(q1+q2+q3+q4)


### Local derivatives through `PartialD(...)`

`PartialD(...)` is the declarative front end for ordinary local derivatives.
It is not a gauge-covariant derivative. It just marks which field the ordinary
partial derivative acts on inside a local monomial.

Typical patterns are:

- `PartialD(Phi, mu) * PartialD(Phi, mu) * Phi * Phi`
- `Psi.bar * Gamma(mu) * PartialD(Psi, nu) * Phi * Chi`
- nested ordinary derivatives such as `PartialD(PartialD(Phi, mu), nu)` when needed


In [58]:
# Local fermion derivative: PartialD(...) also works in ordinary fermion operators.
mu, nu = S("mu"), S("nu")
Psi = Field(
    "Psi",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=S("psi"),
    conjugate_symbol=S("psibar"),
    indices=(SPINOR_INDEX,),
)
Phi = Field("Phi", spin=0, self_conjugate=True, symbol=S("phi"))
Chi = Field("Chi", spin=0, self_conjugate=True, symbol=S("chi"))
MODEL_LOCAL_FERMION_DERIV = Model(
    fields=(Psi, Phi, Chi),
    lagrangian_decl=S("g_local") * Psi.bar * Gamma(mu) * PartialD(Psi, nu) * Phi * Chi,
)
L_local_fermion_deriv = MODEL_LOCAL_FERMION_DERIV.lagrangian()

print("Source declaration:", MODEL_LOCAL_FERMION_DERIV.lagrangian_decl)
show(
    "Local fermion derivative psibar d psi phi chi",
    L_local_fermion_deriv.feynman_rule(Psi.bar, Psi, Phi, Chi),
)


Source declaration: g_local * Psi.bar * Gamma(mu) * PartialD(Psi, nu) * Phi * Chi

=== Local fermion derivative psibar d psi phi chi ===
g_local*(2*𝜋)^d*gamma(bis(4, i1),bis(4, i2),mink(4, mu))*pcomp(q2,nu)*Delta(q1+q2+q3+q4)


In [59]:
# Dressed covariant operator: the same source term yields a derivative vertex and a gauge-current vertex.
Phi = Field("Phi", spin=0, self_conjugate=True, symbol=S("phi"))
Psi = Field(
    "Psi",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=S("psi"),
    conjugate_symbol=S("psibar"),
    indices=(SPINOR_INDEX,),
    quantum_numbers={"Q": S("Qpsi")},
)
A = Field("A", spin=1, self_conjugate=True, symbol=S("A"), indices=(LORENTZ_INDEX,))
U1 = GaugeGroup(name="U1", abelian=True, coupling=S("e"), gauge_boson=A.symbol, charge="Q")
mu = S("mu")
MODEL_DRESSED = Model(
    gauge_groups=(U1,),
    fields=(Psi, Phi, A),
    lagrangian_decl=I * Psi.bar * Gamma(mu) * CovD(Psi, mu) * Phi * Phi,
)
L_dressed = MODEL_DRESSED.lagrangian()

print("Source declaration:", MODEL_DRESSED.lagrangian_decl)
show("Dressed operator: psibar d psi phi phi", L_dressed.feynman_rule(Psi.bar, Psi, Phi, Phi))
show("Dressed operator: psibar psi A phi phi", L_dressed.feynman_rule(Psi.bar, Psi, A, Phi, Phi))


Source declaration: 1𝑖 * Psi.bar * Gamma(mu) * CovD(Psi, mu) * Phi * Phi

=== Dressed operator: psibar d psi phi phi ===
2𝑖*(2*𝜋)^d*gamma(bis(4, i1),bis(4, i2),mink(4, mu))*pcomp(q2,mu)*Delta(q1+q2+q3+q4)

=== Dressed operator: psibar psi A phi phi ===
-2𝑖*Qpsi*e*(2*𝜋)^d*gamma(bis(4, i1),bis(4, i2),mink(4, i3))*Delta(q1+q2+q3+q4+q5)


In [60]:
# Gauge fixing: use the typed wrapper for the ordinary linear covariant gauge term.
B = Field("B", spin=1, self_conjugate=True, symbol=S("B"), indices=(LORENTZ_INDEX,))
U1GF = GaugeGroup(name="U1GF", abelian=True, coupling=S("eGF"), gauge_boson=B.symbol, charge="Q")
MODEL_GF = Model(
    gauge_groups=(U1GF,),
    fields=(B,),
    lagrangian_decl=GaugeFixing(U1GF, xi=S("xi")),
)
L_gf = MODEL_GF.lagrangian()

print("Source declaration:", MODEL_GF.lagrangian_decl)
show("Gauge-fixing bilinear A A", L_gf.feynman_rule(B, B))


Source declaration: GaugeFixing(U1GF, xi=xi)

=== Gauge-fixing bilinear A A ===
1𝑖*(2*𝜋)^d*pcomp(q1,i1)*pcomp(q2,i2)*Delta(q1+q2)/xi


In [61]:
# Ghosts: use the typed wrapper for the ordinary Faddeev-Popov sector.
MODEL_GHOST = Model(
    gauge_groups=(QCD_GROUP_GHOST,),
    fields=(GluonField, GhostGluonField),
    lagrangian_decl=GhostLagrangian(QCD_GROUP_GHOST),
)
L_ghost = MODEL_GHOST.lagrangian()

print("Source declaration:", MODEL_GHOST.lagrangian_decl)
show("Ghost bilinear cbar c", L_ghost.feynman_rule(GhostGluonField.bar, GhostGluonField))
show("Ghost-gluon vertex cbar G c", L_ghost.feynman_rule(GhostGluonField.bar, GluonField, GhostGluonField))


Source declaration: GhostLagrangian(SU3C)

=== Ghost bilinear cbar c ===
-1𝑖*(2*𝜋)^d*g(coad(8, i1),coad(8, i2))*pcomp(q1,mu)*pcomp(q2,mu)*Delta(q1+q2)

=== Ghost-gluon vertex cbar G c ===
-gS*(2*𝜋)^d*f(coad(8, i1),coad(8, i3),coad(8, i4))*pcomp(q1,i2)*Delta(q1+q2+q3)


In [62]:
# Yang-Mills: FieldStrength^2 generates the pure-gauge sector.
muYM, nuYM = S("muYM", "nuYM")
MODEL_YM = Model(
    gauge_groups=(QCD_GROUP,),
    fields=(GluonField,),
    lagrangian_decl=-(Expression.num(1) / Expression.num(4)) * FieldStrength(QCD_GROUP, muYM, nuYM) * FieldStrength(QCD_GROUP, muYM, nuYM),
)
L_ym = MODEL_YM.lagrangian()

print("Source declaration:", MODEL_YM.lagrangian_decl)
show("Yang-Mills cubic vertex G G G", L_ym.feynman_rule(GluonField, GluonField, GluonField))
show("Yang-Mills quartic vertex G G G G", L_ym.feynman_rule(GluonField, GluonField, GluonField, GluonField))


Source declaration: -1/4 * FieldStrength(SU3C, muYM, nuYM) * FieldStrength(SU3C, muYM, nuYM)

=== Yang-Mills cubic vertex G G G ===
-gS*(2*𝜋)^d*g(mink(4, i1),mink(4, i3))*f(coad(8, i2),coad(8, i4),coad(8, i6))*pcomp(q1,i5)*Delta(q1+q2+q3)+gS*(2*𝜋)^d*g(mink(4, i1),mink(4, i3))*f(coad(8, i2),coad(8, i4),coad(8, i6))*pcomp(q2,i5)*Delta(q1+q2+q3)+gS*(2*𝜋)^d*g(mink(4, i1),mink(4, i5))*f(coad(8, i2),coad(8, i4),coad(8, i6))*pcomp(q1,i3)*Delta(q1+q2+q3)-gS*(2*𝜋)^d*g(mink(4, i1),mink(4, i5))*f(coad(8, i2),coad(8, i4),coad(8, i6))*pcomp(q3,i3)*Delta(q1+q2+q3)-gS*(2*𝜋)^d*g(mink(4, i3),mink(4, i5))*f(coad(8, i2),coad(8, i4),coad(8, i6))*pcomp(q2,i1)*Delta(q1+q2+q3)+gS*(2*𝜋)^d*g(mink(4, i3),mink(4, i5))*f(coad(8, i2),coad(8, i4),coad(8, i6))*pcomp(q3,i1)*Delta(q1+q2+q3)

=== Yang-Mills quartic vertex G G G G ===
-1𝑖*gS^2*(2*𝜋)^d*g(mink(4, i1),mink(4, i3))*g(mink(4, i7),mink(4, i5))*f(coad(8, i2),coad(8, i8),coad(8, color_adj_mid_G_SU3C))*f(coad(8, i4),coad(8, i6),coad(8, color_adj_mid_G_SU3C))*Del

In [63]:
# Full gauge-fixed QCD sector: combine pure gauge, gauge fixing, and ghosts in one declaration.
muQCD, nuQCD = S("muQCD", "nuQCD")
MODEL_QCD_FULL = Model(
    gauge_groups=(QCD_GROUP_GHOST,),
    fields=(GluonField, GhostGluonField),
    lagrangian_decl=(
        -(Expression.num(1) / Expression.num(4)) * FieldStrength(QCD_GROUP_GHOST, muQCD, nuQCD) * FieldStrength(QCD_GROUP_GHOST, muQCD, nuQCD)
        + GaugeFixing(QCD_GROUP_GHOST, xi=xiQCD)
        + GhostLagrangian(QCD_GROUP_GHOST)
    ),
)
L_qcd_full = MODEL_QCD_FULL.lagrangian()

print("Source declaration:", MODEL_QCD_FULL.lagrangian_decl)
show("Gauge-fixed gluon bilinear", L_qcd_full.feynman_rule(GluonField, GluonField))
show("Three-gluon vertex", L_qcd_full.feynman_rule(GluonField, GluonField, GluonField))
show("Ghost-gluon vertex", L_qcd_full.feynman_rule(GhostGluonField.bar, GluonField, GhostGluonField))


Source declaration: -1/4 * FieldStrength(SU3C, muQCD, nuQCD) * FieldStrength(SU3C, muQCD, nuQCD) + GaugeFixing(SU3C, xi=xiQCD) + GhostLagrangian(SU3C)

=== Gauge-fixed gluon bilinear ===
1𝑖*(2*𝜋)^d*g(coad(8, i2),coad(8, i4))*pcomp(q1,i1)*pcomp(q2,i3)*Delta(q1+q2)/xiQCD+1𝑖*(2*𝜋)^d*g(mink(4, i1),mink(4, i3))*g(coad(8, i2),coad(8, i4))*pcomp(q1,rho_G_SU3C)*pcomp(q2,rho_G_SU3C)*Delta(q1+q2)-1𝑖*(2*𝜋)^d*g(coad(8, i2),coad(8, i4))*pcomp(q1,i3)*pcomp(q2,i1)*Delta(q1+q2)

=== Three-gluon vertex ===
-gS*(2*𝜋)^d*g(mink(4, i1),mink(4, i3))*f(coad(8, i2),coad(8, i4),coad(8, i6))*pcomp(q1,i5)*Delta(q1+q2+q3)+gS*(2*𝜋)^d*g(mink(4, i1),mink(4, i3))*f(coad(8, i2),coad(8, i4),coad(8, i6))*pcomp(q2,i5)*Delta(q1+q2+q3)+gS*(2*𝜋)^d*g(mink(4, i1),mink(4, i5))*f(coad(8, i2),coad(8, i4),coad(8, i6))*pcomp(q1,i3)*Delta(q1+q2+q3)-gS*(2*𝜋)^d*g(mink(4, i1),mink(4, i5))*f(coad(8, i2),coad(8, i4),coad(8, i6))*pcomp(q3,i3)*Delta(q1+q2+q3)-gS*(2*𝜋)^d*g(mink(4, i3),mink(4, i5))*f(coad(8, i2),coad(8, i4),coad(8, i6))*pco

In [64]:
# SM slice: the same source term yields the derivative piece plus the gluon and photon currents.
# With identical Phi * Phi spectators, the vertices carry the expected combinatorial factor 2.
muSM = S("muSM")
Phi = Field("Phi", spin=0, self_conjugate=True, symbol=S("Phi"))
qSM = Field(
    "qSM",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=S("qSM0"),
    conjugate_symbol=S("qSMbar0"),
    indices=(SPINOR_INDEX, COLOR_FUND_INDEX),
    quantum_numbers={"Q": S("QqSM")},
)
MODEL_SM_SLICE = Model(
    gauge_groups=(QCD_GROUP, QED_GROUP),
    fields=(qSM, Phi, GluonField, GaugeField),
    lagrangian_decl=I * qSM.bar * Gamma(muSM) * CovD(qSM, muSM) * Phi * Phi,
)
L_sm = MODEL_SM_SLICE.lagrangian()

print("Source declaration:", MODEL_SM_SLICE.lagrangian_decl)
show("SM slice: qbar q Phi Phi", L_sm.feynman_rule(qSM.bar, qSM, Phi, Phi))
show("SM slice: qbar q G Phi Phi", L_sm.feynman_rule(qSM.bar, qSM, GluonField, Phi, Phi))
show("SM slice: qbar q A Phi Phi", L_sm.feynman_rule(qSM.bar, qSM, GaugeField, Phi, Phi))


Source declaration: 1𝑖 * qSM.bar * Gamma(muSM) * CovD(qSM, muSM) * Phi * Phi

=== SM slice: qbar q Phi Phi ===
2𝑖*(2*𝜋)^d*g(cof(3, i2),cof(3, i4))*gamma(bis(4, i1),bis(4, i3),mink(4, mu))*pcomp(q2,mu)*Delta(q1+q2+q3+q4)

=== SM slice: qbar q G Phi Phi ===
-2𝑖*gS*(2*𝜋)^d*gamma(bis(4, i1),bis(4, i3),mink(4, i5))*t(coad(8, i6),cof(3, i2),cof(3, i4))*Delta(q1+q2+q3+q4+q5)

=== SM slice: qbar q A Phi Phi ===
-2𝑖*eQED*QqSM*(2*𝜋)^d*g(cof(3, i2),cof(3, i4))*gamma(bis(4, i1),bis(4, i3),mink(4, i5))*Delta(q1+q2+q3+q4+q5)


## 12. Current coverage

Recommended workflow:

1. define metadata
2. write `lagrangian_decl=...`
3. call `Model.lagrangian()`
4. ask for the vertex you want with `feynman_rule(...)`

Implemented declarative coverage:

- local monomials and sums of local monomials
- ordinary local derivative monomials through `PartialD(...)`, including nested ordinary derivatives
- canonical fermion and complex-scalar covariant terms through `CovD(...)`
- dressed covariant operators with extra local spectators
- pure-gauge Yang-Mills terms through `FieldStrength(...)`
- linear covariant gauge fixing through `GaugeFixing(...)`
- non-abelian ghosts through `GhostLagrangian(...)`
- mixed `SU(3)_c x U(1)` models in one declaration

Current limits:

- there is no free-form symbolic parser yet; only the supported canonical patterns expand automatically
- `feynman_rule(...)` extracts one chosen vertex, not yet a whole-Lagrangian `FeynmanRules[...]` equivalent
- very exotic local tensor structures may still need `InteractionTerm(...)`
- repeated same-kind gauge slots may still need explicit `slot_policy` choices
- the full electroweak Standard Model and background-field layer are still future work

For the internals, start with `src/model/`, `src/compiler/gauge.py`, `src/compiler/gauge.py`, and `src/symbolic/vertex_engine.py`.


## 13. What to use for full regression coverage

This notebook is a front-end walkthrough, not the full validation matrix.

For exhaustive executable coverage use:

- `examples/examples_lagrangian.py` for declarative demos
- `examples/examples.py` for the broader reference suite
- `tests/` for focused API and compiler regressions
